# 00 · Colab Setup

Mount Drive, configure paths, install deps, and download the Kaggle dataset.
**Run this once per Colab session before any other notebook.**

Dataset: [pkdarabi/cardetection](https://www.kaggle.com/datasets/pkdarabi/cardetection/data) — *Traffic Signs Detection* (15 classes, required teacher dataset).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo from GitHub + make `src` importable

In [ ]:
import os, sys, subprocess

GITHUB_REPO = 'https://github.com/huynhphtloi/traffic-sign-detection-yolo-detr'
DRIVE_ROOT = '/content/drive/MyDrive/HCMUE_Projects/DeepLearning'
REPO      = '/content/traffic-sign-detection-yolo-detr'
DATA_DIR  = f'{DRIVE_ROOT}/data'

if not os.path.exists(REPO):
    result = subprocess.run(['git', 'clone', GITHUB_REPO, REPO], capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f'git clone failed — check the repo URL or network access')

if not os.path.exists(f'{REPO}/src'):
    raise RuntimeError(f'Clone succeeded but src/ not found in {REPO} — check the repo structure')

os.environ['TSD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['TSD_REPO_ROOT']  = REPO
os.environ['TSD_DATA_DIR']   = DATA_DIR
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('drive root:', DRIVE_ROOT)
print('repo root :', REPO)
print('data dir  :', DATA_DIR)
print('src found :', os.path.exists(f'{REPO}/src'))

## 3. Install dependencies

In [ ]:
import os
REPO = os.environ['TSD_REPO_ROOT']
!pip install -q -r {REPO}/requirements.txt

## 4. Kaggle credentials

`kaggle.json` is stored in `MyDrive/HCMUE_Projects/DeepLearning/` on Drive (outside the repo — never commit it). We copy it to `/root/.kaggle/` where the Kaggle CLI expects it.

In [ ]:
import os

DRIVE_ROOT = os.environ['TSD_DRIVE_ROOT']
os.makedirs('/root/.kaggle', exist_ok=True)
!cp {DRIVE_ROOT}/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('kaggle.json installed')

## 5. Download the dataset

In [ ]:
import os

DATA_DIR = os.environ['TSD_DATA_DIR']
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
!kaggle datasets download -d pkdarabi/cardetection -p {DATA_DIR}/raw --unzip
!ls {DATA_DIR}/raw

## 6. Organise into `data/processed/cardetection/`

The Roboflow export unzips to a single subfolder (e.g. `data/raw/cardetection/` or similar). This cell finds it and symlinks it to `data/processed/cardetection` so all scripts resolve to the same root.

In [ ]:
import os, pathlib

DATA_DIR = pathlib.Path(os.environ['TSD_DATA_DIR'])
raw = DATA_DIR / 'raw'
processed = DATA_DIR / 'processed' / 'cardetection'

# Find the subfolder that contains data.yaml (the Roboflow export root).
export_root = None
for candidate in [raw, *raw.iterdir()]:
    if (candidate / 'data.yaml').exists():
        export_root = candidate
        break

if export_root is None:
    raise FileNotFoundError('Could not locate data.yaml under data/raw — check the download.')

processed.parent.mkdir(parents=True, exist_ok=True)
if not processed.exists():
    processed.symlink_to(export_root.resolve())
    print(f'symlinked {processed} -> {export_root}')
else:
    print(f'already exists: {processed}')

for split in ('train', 'valid', 'test'):
    imgs = list((processed / split / 'images').glob('*')) if (processed / split / 'images').exists() else []
    print(f'  {split}: {len(imgs)} images')

## 7. Verify: inspect the dataset

In [ ]:
import os, pathlib
from src.data.inspect_dataset import inspect

DATA_PROCESSED = pathlib.Path(os.environ['TSD_DATA_DIR']) / 'processed' / 'cardetection'
df = inspect(DATA_PROCESSED)
print(df.to_string(index=False))
print(f"\nformat      : {df['format'].iloc[0]}")
print(f"class_names : {df.attrs.get('class_names')}")